In [ ]:
!pip install datasets bitsandbytes zstd

In [ ]:
# !rm -rf ~/.cache/huggingface/hub/

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import BitsAndBytesConfig
import torch
from datasets import load_dataset
import os
import gc
from tqdm import tqdm
import numpy as np
from collections import defaultdict
import bz2
import urllib.request

In [ ]:
from huggingface_hub import login
from google.colab import userdata

huggingface_token = userdata.get('HF_TOKEN')
login(token=huggingface_token)

In [ ]:
NUM_SAMPLES = 10000
MAX_SAMPLE_CHARS = 6000
VISUALIZATION_N_FIRST_TOKENS = 2000
VISUALIZATION_GAMMA = 2
VISUALIZATION_TOP_NEXT_TOKENS = 3

# MODELS = [
#     'google/gemma-3-12b-it',
#     'INSAIT-Institute/MamayLM-Gemma-3-12B-IT-v1.0',
#     'lapa-llm/lapa-v0.1.2-instruct',
# ]

MODELS = [
    'google/gemma-3-12b-pt',
    'Qwen/Qwen2.5-7B-Instruct',
    'google/gemma-3-1b-pt',
    'lapa-llm/lapa-12b-pt',
    'meta-llama/Llama-3.2-3B',
    # 'meta-llama/Llama-3.1-8B' # No space

] # PT models

In [ ]:
# def load_and_sample_texts(texts, max_chars=MAX_SAMPLE_CHARS, num_samples=NUM_SAMPLES):
#     chunks = []
#     for doc in texts:
#         chunks.extend([doc[i:i+max_chars] for i in range(0, len(doc), max_chars)])
#     return chunks[:num_samples]


import re

MIN_SENTENCE_LENGTH = 120
MAX_SENTENCE_LENGTH = 200

def preprocess_sentences(texts):
    all_text = "".join(texts)
    sentences = re.split(r'[.!?]+', all_text)
    sentences = [s.strip() for s in sentences if s.strip()]

    clean = [s for s in sentences if not any(c.isdigit() or c in 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz' for c in s)]

    filtered = [s for s in clean if MIN_SENTENCE_LENGTH <= len(s) <= MAX_SENTENCE_LENGTH]
    return filtered

ds = load_dataset("a-l-o/news", split="train")
raw_texts = preprocess_sentences(ds["text"])
print(f"{len(raw_texts)} sentences, {sum(len(s) for s in raw_texts)} total chars")

148 sentences, 22362 total chars


In [ ]:
print("Loading dataset...")


from datasets import load_dataset

# print("Loading Formal Speech")
# ds = load_dataset("a-l-o/Formal-speech", split="train")
# print(f"Loaded {len(ds):,} documents")
# raw_texts = load_and_sample_texts(ds["text"])


# print("Loading Medical")
# ds = load_dataset("a-l-o/Medical", split="train")
# print(f"Loaded {len(ds):,} documents")
# raw_texts = load_and_sample_texts(ds["text"])

# print("Loading news")
# ds = load_dataset("a-l-o/news", split="train")
# print(f"Loaded {len(ds):,} documents")
# raw_texts = load_and_sample_texts(ds["text"])

# text = "".join(raw_texts)
# encoded = text.encode('utf-8')
# import zstd

# compressed = zstd.compress(encoded, 22)
# print(f"{len(compressed) * 8 / len(text):.3f} zstd-compressed bits/char")



Loading dataset...


In [ ]:
def tokenize_text(text, tokenizer):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    char_length = len(text)
    byte_length = len(text.encode('utf-8'))
    return tokens, char_length, byte_length

In [ ]:
def calculate_bits_per_token(model, tokenizer, samples, device):
    total_loss = 0
    total_tokens = 0
    total_chars = 0
    total_bytes = 0
    all_tokens = []
    all_scores = []
    all_top_predictions = []
    for tokens, char_len, byte_len in tqdm(samples, desc="Evaluating samples"):
        bos_id = tokenizer.bos_token_id
        if bos_id is not None:
            tokens_with_bos = [bos_id] + tokens
        else:
            tokens_with_bos = tokens
        input_ids = torch.tensor([tokens_with_bos]).to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, labels=input_ids)
            loss_sum = outputs.loss.item() * len(tokens)
            total_loss += loss_sum

            logits = outputs.logits
            shift_logits = logits[:, :-1, :]
            shift_labels = input_ids[:, 1:]

            loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
            per_token_loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

            all_tokens.extend([tokenizer.decode([t]) for t in shift_labels[0].tolist()])
            all_scores.extend(per_token_loss.tolist())


            if len(all_top_predictions) < VISUALIZATION_N_FIRST_TOKENS:
                probs = torch.softmax(shift_logits[0], dim=-1)
                top_probs, top_indices = torch.topk(probs, k=VISUALIZATION_TOP_NEXT_TOKENS, dim=-1)
                remaining = VISUALIZATION_N_FIRST_TOKENS - len(all_top_predictions)
                all_top_predictions.extend([[(tokenizer.decode([idx.item()]), p.item()) for idx, p in zip(pos_idx, pos_prob)] for pos_idx, pos_prob in zip(top_indices[:remaining], top_probs[:remaining])])

            total_tokens += len(tokens)
            total_chars += char_len
            total_bytes += byte_len
            torch.cuda.empty_cache()

    total_bits = total_loss / torch.log(torch.tensor(2.0)).item()
    return {
        'bits_per_token': total_bits / total_tokens,
        'bits_per_char': total_bits / total_chars,
        'bits_per_byte': total_bits / total_bytes,
        'total_tokens': total_tokens,
        'chars_per_token': total_chars / total_tokens,
        'tokens': all_tokens,
        'scores': all_scores,
        'top_predictions': all_top_predictions,
    }

In [ ]:
from IPython.display import HTML, display

def visualize_token_scores(tokens, scores, title="", gamma=1.0, top_predictions=None):
    """
    tokens: list of strings
    scores: list of floats (will be normalized to [-1, 1])
    """
    original_scores = scores.copy()
    original_max = scores.max()
    scores = scores / scores.max()
    scores = scores ** gamma

    html_parts = [f"<div style='font-family: Georgia, serif; line-height: 1.8; padding: 10px;'>"]
    if title:
        html_parts.append(f"<b>{title}</b><br><br>")
    legend_html = "<div style='display: flex; align-items: center; margin-bottom: 10px;'>"
    probs = [100, 75, 50, 25, 10, 1, 0.1, 0.01, 0.001] # Legend numbers
    for prob in probs:
        s = -np.log(prob / 100) / original_max
        s = min(s, 1.0)
        r, g, b = 255, int(255 * (1 - s ** gamma)), int(255 * (1 - s ** gamma))
        legend_html += f"<div style='background-color: rgb({r},{g},{b}); padding: 4px 8px;'>{prob}%</div>"
    legend_html += "</div>"
    html_parts.append(legend_html)
    for i, (tok, score, orig) in enumerate(zip(tokens, scores, original_scores)):
      prob = 100 * np.exp(-orig)
      r, g, b = 255, int(255 * (1 - score ** gamma)), int(255 * (1 - score ** gamma))

      tok_escaped = tok.replace('<', '&lt;').replace('>', '&gt;').replace(' ', '&nbsp;')

      if top_predictions:
          top3 = "\n".join([f"({t}): {p*100:.1f}%" for t, p in top_predictions[i]])
          tooltip = f"Predicted probability of this token: {prob:.4f}% \n\n Top3 predictions: \n {top3}"
      else:
          tooltip = f"{prob:.4f}%"

      html_parts.append(f"<span title='{tooltip}' style='background-color: rgb({r},{g},{b}); padding: 2px 0; cursor: default;'>{tok_escaped}</span>")
    html_parts.append("</div>")
    display(HTML("".join(html_parts)))

In [ ]:
def evaluate_model(model_name, raw_texts):
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16 if "Qwen" not in model_name else torch.float16,
        device_map="auto"
    )
    model.eval()

    print(f"Pad token: {tokenizer.pad_token}")
    print(f"Vocab size: {len(tokenizer)}")
    test_input = tokenizer.encode("Привіт!", return_tensors="pt").to(model.device)
    test_output = model.generate(test_input, max_length=50)
    print(f"Test generation: {tokenizer.decode(test_output[0])}")
    samples = []
    for text in tqdm(raw_texts, desc="Processing texts"):
        tokens, char_len, byte_len = tokenize_text(text, tokenizer)
        samples.append((tokens, char_len, byte_len))
        # print(f"Sample: {len(tokens)} tokens, {char_len} chars, {byte_len} bytes")

    metrics = calculate_bits_per_token(model, tokenizer, samples, model.device)

    print(f"\n🎯 Results:")
    print(f"   {metrics['bits_per_token']:.3f} bits/token")
    print(f"   {metrics['bits_per_char']:.3f} bits/char")
    print(f"   {metrics['bits_per_byte']:.3f} bits/byte")
    print(f"   {metrics['chars_per_token']:.2f} chars/tok")
    print(f"   {metrics['total_tokens']} total tokens evaluated")

    visualize_token_scores(metrics['tokens'][:VISUALIZATION_N_FIRST_TOKENS], np.array(metrics['scores'][:VISUALIZATION_N_FIRST_TOKENS]), title="Per-token loss", gamma=VISUALIZATION_GAMMA, top_predictions=metrics['top_predictions'][:VISUALIZATION_N_FIRST_TOKENS])

    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()

    import time
    time.sleep(5)

    return {
        'model_name': model_name,
        **metrics,
    }

In [ ]:
# Run evaluation for all models
results = []
for model_name in MODELS:
    try:
        result = evaluate_model(model_name, raw_texts)
        results.append(result)
    except Exception as e:
        print(f"❌ Error with {model_name}: {e}")
        results.append({
            'model_name': model_name,
            'bits_per_token': None,
            'error': str(e)
        })

print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
for result in results:
    if result['bits_per_token'] is not None:
        print(f"{result['model_name']}: {result['bits_per_token']:.3f} bpt | {result['bits_per_char']:.3f} bpc | {result['bits_per_byte']:.3f} bpb")
    else:
        print(f"{result['model_name']}: ERROR - {result.get('error', 'Unknown')}")

valid_results = [r for r in results if r['bits_per_token'] is not None]
valid_results.sort(key=lambda x: x['bits_per_byte'])

print(f"\n{'='*60}")
print("RANKING (Best to Worst)")
print(f"{'='*60}")
for i, result in enumerate(valid_results, 1):
    print(f"{i}. {result['model_name']}: {result['bits_per_token']:.3f} bpt | {result['bits_per_char']:.3f} bpc | {result['bits_per_byte']:.3f} bpb")


Evaluating: google/gemma-3-12b-pt


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

Pad token: <pad>
Vocab size: 262145
Test generation: <bos>Привіт! Мене звати Максим Мотовилов. Я інтернет-маркетолог, який працює з сайтами та веб-сторінками. З 2020 року я допомагаю організаціям


Evaluating samples: 100%|██████████| 148/148 [00:16<00:00,  9.09it/s]

...

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Pad token: None
Vocab size: 128256
Test generation: <|begin_of_text|>Привіт! Добро пожаловать! Welcome! Bienvenue! Benvenuti! Willkommen! Bem-vindo! Bienvenido! Bienvenido! Bienvenido! Bienvenido! Bienvenido! Welcome!


Evaluating samples: 100%|██████████| 148/148 [00:06<00:00, 24.50it/s]



🎯 Results:
   4.645 bits/token
   1.499 bits/char
   0.815 bits/byte
   3.10 chars/tok
   7215 total tokens evaluated



SUMMARY
google/gemma-3-12b-pt: 3.630 bpt | 1.168 bpc | 0.635 bpb
Qwen/Qwen2.5-7B-Instruct: 3.129 bpt | 1.558 bpc | 0.847 bpb
google/gemma-3-1b-pt: 4.335 bpt | 1.395 bpc | 0.758 bpb
lapa-llm/lapa-12b-pt: 5.419 bpt | 1.072 bpc | 0.583 bpb
meta-llama/Llama-3.2-3B: 4.645 bpt | 1.499 bpc | 0.815 bpb

RANKING (Best to Worst)
1. lapa-llm/lapa-12b-pt: 5.419 bpt | 1.072 bpc | 0.583 bpb
2. google/gemma-3-12b-pt: 3.630 bpt | 1.168 bpc | 0.635 bpb
3. google/gemma-3-1b-pt: 4.335 bpt | 1.395 bpc | 0.758 bpb
4. meta-llama/Llama-3.2-3B: 4.645 bpt | 1.499 bpc | 0.815 bpb
5. Qwen/Qwen2.5-7B-Instruct: 3.129 bpt | 1.558 bpc | 0.847 bpb


## Dataset diversity check (reviewer 2)

Concern: 5 news articles may share topical vocabulary ("topical salience") that makes characters artificially predictable, biasing entropy downward.

We report pairwise Jaccard similarity on word sets and $n$-gram uniqueness (fraction of distinct $n$-grams that appear in only one article) across the 5 source articles.

In [ ]:
import re
import itertools
from collections import Counter
from datasets import load_dataset

UK_LETTERS = set('абвгґдеєжзиіїйклмнопрстуфхцчшщьюя')

def clean_text(text):
    text = text.lower()
    cleaned = ''.join(c if c in UK_LETTERS or c.isspace() else ' ' for c in text)
    return re.sub(r'\s+', ' ', cleaned).strip()

ds = load_dataset("a-l-o/shortnews", split="train")
articles = ds["text"]
cleaned = [clean_text(a) for a in articles]
print(f"Articles: {len(cleaned)}")
for i, a in enumerate(cleaned):
    print(f"  Article {i+1}: {len(a):6d} chars, {len(a.split()):5d} words")

# Pairwise Jaccard on word sets
print("\n=== Pairwise Jaccard (word sets) ===")
word_sets = [set(a.split()) for a in cleaned]
pairs = []
for i, j in itertools.combinations(range(len(cleaned)), 2):
    inter = len(word_sets[i] & word_sets[j])
    union = len(word_sets[i] | word_sets[j])
    jac = inter / union if union else 0.0
    pairs.append(jac)
    print(f"  {i+1} vs {j+1}: J = {jac:.4f}  (|∩|={inter}, |∪|={union})")
print(f"  Mean pairwise Jaccard: {sum(pairs)/len(pairs):.4f}")
print(f"  Min / Max:             {min(pairs):.4f} / {max(pairs):.4f}")

# N-gram uniqueness: fraction of distinct n-grams found in only one article
def word_ngrams(text, n):
    w = text.split()
    return [tuple(w[i:i+n]) for i in range(len(w) - n + 1)]

print("\n=== N-gram uniqueness (fraction of distinct n-grams appearing in only 1 article) ===")
print(f"{'n':>3} {'Distinct':>10} {'Only in 1':>10} {'Unique%':>9}")
for n in [1, 2, 3, 4, 5]:
    presence = Counter()
    for a in cleaned:
        for ng in set(word_ngrams(a, n)):
            presence[ng] += 1
    distinct = len(presence)
    unique = sum(1 for v in presence.values() if v == 1)
    print(f"{n:>3} {distinct:>10d} {unique:>10d} {unique/distinct:>8.1%}")


README.md:   0%|          | 0.00/267 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/63.0k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5 [00:00<?, ? examples/s]

Articles: 5
  Article 1:  10349 chars,  1495 words
  Article 2:  12183 chars,  1793 words
  Article 3:  17258 chars,  2357 words
  Article 4:   9361 chars,  1318 words
  Article 5:  16713 chars,  2468 words

=== Pairwise Jaccard (word sets) ===
  1 vs 2: J = 0.0807  (|∩|=125, |∪|=1549)
  1 vs 3: J = 0.0964  (|∩|=178, |∪|=1847)
  1 vs 4: J = 0.1113  (|∩|=159, |∪|=1428)
  1 vs 5: J = 0.0744  (|∩|=149, |∪|=2003)
  2 vs 3: J = 0.0868  (|∩|=159, |∪|=1832)
  2 vs 4: J = 0.0845  (|∩|=121, |∪|=1432)
  2 vs 5: J = 0.1072  (|∩|=205, |∪|=1913)
  3 vs 4: J = 0.1115  (|∩|=191, |∪|=1713)
  3 vs 5: J = 0.0944  (|∩|=213, |∪|=2256)
  4 vs 5: J = 0.0815  (|∩|=153, |∪|=1878)
  Mean pairwise Jaccard: 0.0929
  Min / Max:             0.0744 / 0.1115

=== N-gram uniqueness (fraction of distinct n-grams appearing in only 1 article) ===
  n   Distinct  Only in 1   Unique%
  1       3813       3143    82.4%
  2       8484       8255    97.3%
  3       9270       9239    99.7%
  4       9375       9364    99.9%


In [ ]:
for i, a in enumerate(articles, 1):
    print(f"--- Article {i} ({len(a)} chars) ---")
    print(a[:400].strip())
    print()
